# 구조화된 출력 (Structured Output)

구조화된 출력을 사용하면 에이전트가 특정하고 예측 가능한 형식으로 데이터를 반환할 수 있습니다. 자연어 응답을 구문 분석하는 대신 애플리케이션에서 직접 사용할 수 있는 JSON 객체, Pydantic 모델 또는 데이터클래스 형태로 구조화된 데이터를 얻을 수 있습니다.

**구조화된 출력 전략:**

| 전략 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 구조화된 출력 지원 모델용 |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **자동 선택** | 스키마 타입만 전달 시 모델에 따라 최적 전략 자동 선택 |

LangChain의 `create_agent`는 `response_format` 매개변수로 구조화된 출력을 설정하며, 결과는 에이전트 상태의 `structured_response` 키에 반환됩니다.

> 📖 **참고 문서**: [LangChain Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

---

## Response Format

에이전트가 구조화된 데이터를 반환하는 방법을 `response_format` 매개변수로 제어합니다:

| 설정 | 설명 |
|:---|:---|
| **ToolStrategy[T]** | 도구 호출을 통한 구조화된 출력 |
| **ProviderStrategy[T]** | 제공자 네이티브 구조화된 출력 사용 |
| **type[T]** | 스키마 타입 직접 전달 - 모델에 따라 최적 전략 자동 선택 |
| **None** | 구조화된 출력 없음 |

스키마 타입이 직접 제공되면 LangChain이 자동으로 최적의 전략을 선택합니다:
- 네이티브 구조화된 출력 지원 모델(OpenAI, Anthropic, Grok)에는 `ProviderStrategy`
- 다른 모든 모델에는 `ToolStrategy`

구조화된 응답은 에이전트의 최종 상태의 `structured_response` 키에 반환됩니다.

---

## 오류 처리

모델은 도구 호출을 통해 구조화된 출력을 생성할 때 스키마와 일치하지 않는 값을 반환할 수 있습니다. 예를 들어, 1-5 범위로 지정된 rating에 10을 반환하거나, 필수 필드를 누락하는 경우가 있습니다. LangChain은 이러한 오류를 자동으로 감지하고 처리하는 지능형 재시도 메커니즘을 제공합니다.

`handle_errors` 매개변수로 오류 처리 방법을 세밀하게 제어할 수 있으며, 기본값은 `True`로 모든 검증 오류를 자동으로 처리합니다. 오류가 발생하면 에이전트는 모델에게 오류 내용을 피드백하고 재시도를 요청하여, 최종적으로 스키마에 맞는 출력을 얻도록 합니다.

이 기능은 프로덕션 환경에서 안정적인 구조화된 출력을 보장하는 데 필수적입니다.

### 스키마 검증 오류

구조화된 출력이 예상 스키마와 일치하지 않으면 에이전트는 오류 피드백을 제공하고 모델에게 재시도를 요청합니다. 예를 들어, rating이 1-5 범위인데 사용자가 "10/10"이라고 입력하면, 모델이 이를 10으로 파싱하려다 검증 오류가 발생하고 자동으로 5로 수정됩니다.

이 재시도 메커니즘 덕분에 실제 사용자 입력이 예상 범위를 벗어나더라도 안정적으로 처리할 수 있습니다. LangSmith에서 실행 추적을 확인하면 재시도 과정을 시각적으로 볼 수 있습니다.

아래 코드는 스키마 검증 오류가 발생했을 때 자동으로 수정되는 예시입니다.

In [2]:
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="default")

# 10/10 입력 - 범위를 벗어나므로 자동으로 5로 수정됨
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)

# 구조화된 응답 확인
print(result["structured_response"])
# ProductRating(rating=5, comment='Amazing product')
# 모델이 자동으로 수정하여 10을 5로 변경


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_5FosoSQdCBLUuAA0vZRw5Qdx)
 Call ID: call_5FosoSQdCBLUuAA0vZRw5Qdx
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'
rating=5 comment='Amazing product, 10/10!'


### 오류 처리 전략

`handle_errors` 매개변수를 사용하여 오류 처리 방법을 다양하게 커스터마이징할 수 있습니다. 요구사항에 따라 자동 재시도, 예외 발생, 커스텀 핸들러 등을 선택할 수 있습니다.

| 설정 | 설명 |
|:---|:---|
| `True` | 모든 오류를 자동으로 처리하고 재시도 (기본값) |
| `False` | 오류 발생 시 예외 발생 |
| 문자열 | 커스텀 오류 메시지로 재시도 |
| 예외 클래스 | 특정 예외만 처리 |
| 콜러블 | 커스텀 오류 핸들러 함수 사용 |

아래에서 각 전략의 사용 예시를 살펴봅니다.

### 커스텀 오류 메시지

문자열을 `handle_errors`에 전달하면 해당 메시지로 모델에게 재시도를 요청합니다. 기본 오류 메시지 대신 더 구체적인 안내를 제공할 수 있으므로, 모델이 오류를 더 잘 이해하고 올바른 형식으로 수정할 수 있습니다.

특히 복잡한 스키마나 특정 도메인의 규칙이 있는 경우, 커스텀 메시지로 명확한 지시를 제공하면 재시도 성공률이 높아집니다.

아래 코드는 커스텀 오류 메시지를 설정하는 예시입니다.

In [3]:
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="custom_message")

# 커스텀 오류 메시지로 에이전트 생성
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_SzgE7EfUK0xi3NBiVtOYzRDp)
 Call ID: call_SzgE7EfUK0xi3NBiVtOYzRDp
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'


### 특정 예외만 처리

예외 클래스를 `handle_errors`에 전달하면 해당 예외만 처리하고 다른 예외는 그대로 발생합니다. 이는 예상된 검증 오류만 재시도하고, 예상치 못한 시스템 오류는 즉시 감지하고 싶을 때 유용합니다.

예를 들어, `ValueError`만 처리하면 범위 검증 오류는 재시도하지만, 네트워크 오류나 타입 불일치 같은 다른 오류는 예외로 발생시킵니다.

아래 코드는 특정 예외만 처리하는 예시입니다.

In [4]:
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="value_error_only")

# 커스텀 오류 메시지로 에이전트 생성
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_cVfQRGJWE228aqQvngdQ7RNh)
 Call ID: call_cVfQRGJWE228aqQvngdQ7RNh
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'


### 여러 예외 유형 처리

튜플로 여러 예외 클래스를 `handle_errors`에 전달하면 해당 예외들을 모두 처리합니다. 이는 여러 종류의 검증 오류를 한 번에 처리하고 싶을 때 유용합니다.

일반적으로 `ValueError`와 `TypeError`를 함께 처리하면 대부분의 스키마 검증 오류를 커버할 수 있습니다.

아래 코드는 여러 예외 유형을 처리하는 예시입니다.

In [5]:
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="multiple_exceptions")

# 커스텀 오류 메시지로 에이전트 생성
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_7ddG0RB7QX8LqTnP1H1WilvR)
 Call ID: call_7ddG0RB7QX8LqTnP1H1WilvR
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'


### 커스텀 오류 핸들러 함수

함수(콜러블)를 `handle_errors`에 전달하면 오류 발생 시 해당 함수가 호출되어 커스텀 오류 메시지를 생성합니다. 이 방법은 오류 유형에 따라 다른 메시지를 반환하거나, 오류 정보를 로깅하는 등 복잡한 처리가 필요할 때 유용합니다.

LangChain은 `StructuredOutputValidationError`(스키마 검증 실패)와 `MultipleStructuredOutputsError`(여러 출력 반환) 등의 구체적인 예외 클래스를 제공합니다. 이를 활용하면 오류 유형별로 세밀한 처리가 가능합니다.

아래 코드는 커스텀 오류 핸들러 함수를 사용하는 예시입니다.

In [6]:
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="custom_handler")

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_TI0E6ldaZDbbnBm73xFGUiok)
 Call ID: call_TI0E6ldaZDbbnBm73xFGUiok
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'


### 오류 처리 비활성화

`handle_errors=False`를 설정하면 오류 발생 시 예외가 그대로 발생합니다. 이는 개발 중에 오류를 디버깅하거나, 커스텀 오류 처리 로직을 직접 구현하고 싶을 때 유용합니다.

프로덕션 환경에서는 일반적으로 `handle_errors=True`(기본값)를 사용하여 안정적인 동작을 보장하지만, 개발 및 테스트 환경에서는 비활성화하여 문제를 빠르게 파악할 수 있습니다.

아래 코드는 오류 처리를 비활성화하는 예시입니다.

In [7]:
# 오류 처리 비활성화 - 모든 오류가 예외로 발생
from feature.structured_output.AgentStructuredOutputHandleError import AgentStructuredOutputHandleErrorAgent

workflow = AgentStructuredOutputHandleErrorAgent(error_strategy="disabled")

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_DGbu6PLdGooiGCOlecvHp6Pu)
 Call ID: call_DGbu6PLdGooiGCOlecvHp6Pu
  Args:
    rating: 5
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'
structured_response:
rating=5 comment='Amazing product, 10/10!'


---

## 종합 예제

지금까지 학습한 `Union` 타입과 오류 처리를 결합한 실용적인 예제입니다. 하나의 에이전트가 책과 영화 추천을 모두 처리하며, 입력 내용에 따라 적절한 스키마를 자동으로 선택합니다.

`handle_errors=True`로 설정하여 오류 발생 시 자동으로 재시도하므로, 프로덕션 환경에서도 안정적으로 동작합니다. 이 패턴은 다양한 유형의 입력을 처리해야 하는 챗봇이나 AI 어시스턴트에 적합합니다.

아래 코드는 책과 영화 추천을 처리하는 에이전트 예시입니다.

In [14]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class BookRecommendation(BaseModel):
    """책 추천 정보를 나타내는 스키마
    
    제목, 저자, 장르, 평점, 요약을 구조화합니다.
    """

    title: str = Field(description="Book title")  # 책 제목
    author: str = Field(description="Author name")  # 저자명
    genre: Literal["fiction", "non-fiction", "science", "history", "biography"] = Field(
        description="Book genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)
    summary: str = Field(description="Brief summary of the book")  # 책 요약


class MovieRecommendation(BaseModel):
    """영화 추천 정보를 나타내는 스키마
    
    제목, 감독, 개봉년도, 장르, 평점을 구조화합니다.
    """

    title: str = Field(description="Movie title")  # 영화 제목
    director: str = Field(description="Director name")  # 감독명
    year: int = Field(description="Release year")  # 개봉년도
    genre: Literal["action", "comedy", "drama", "horror", "sci-fi"] = Field(
        description="Movie genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)


# 에이전트 생성 - Union 타입으로 여러 추천 유형 지원
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=Union[BookRecommendation, MovieRecommendation], handle_errors=True
    ),
    system_prompt="You are a helpful entertainment recommendation assistant.",
)

# 책 추천 요청
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Recommend a good science fiction book"}]}
)
print("Book recommendation:")
print(result1["structured_response"])

# 영화 추천 요청
result2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend a comedy movie from the 2000s"}
        ]
    }
)
print("\nMovie recommendation:")
print(result2["structured_response"])

Book recommendation:
title='Dune' author='Frank Herbert' genre='science' rating=5 summary='Set in the distant future, Dune follows Paul Atreides as his family takes control of the desert planet Arrakis, the only source of the valuable spice melange. The story weaves together themes of politics, religion, ecology, and destiny as Paul navigates betrayal, finds his place among the native Fremen, and potentially fulfills an ancient prophecy. This epic tale explores power, survival, and human potential in a richly imagined universe.'

Movie recommendation:
title='Superbad' director='Greg Mottola' year=2007 genre='comedy' rating=4


---

## 정리

이 튜토리얼에서는 LangGraph 에이전트의 구조화된 출력 기능을 학습했습니다. 구조화된 출력을 사용하면 에이전트의 응답을 예측 가능한 형식으로 받아 애플리케이션에서 쉽게 처리할 수 있습니다.

**핵심 개념 요약:**

| 개념 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 지원 모델용 (가장 신뢰성 높음) |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **Union 타입** | 여러 스키마 중 자동 선택 |
| **handle_errors** | 오류 발생 시 자동 재시도 및 수정 |

**스키마 정의 방법:**
- **Pydantic 모델**: 가장 풍부한 검증 기능 제공 (권장)
- **데이터클래스**: 간단한 스키마 정의, 추가 의존성 없음
- **TypedDict**: 딕셔너리 형태로 반환, JSON 직렬화 용이

**실전 팁:**
- 필드에 명확한 `description` 제공
- `Literal` 타입으로 허용값 제한
- `ge`, `le` 등 검증자로 범위 지정
- `Union` 타입으로 다양한 응답 유형 처리
- 프로덕션에서는 `handle_errors=True` 사용 (기본값)